In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import tensorflow_addons as tfa

# Load data
df = pd.read_csv("../../data/processed/sfax_weather_features.csv", parse_dates=['valid_time'])
df = df.set_index('valid_time')

# Scaling
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df[['t2m']])

# Create supervised sequences
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 24
X, y = create_sequences(df_scaled, seq_length)

# Train/Validation split
train_size = df.loc[:'2023-12-31'].shape[0] - seq_length
X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:], y[train_size:]

# TCN Model
model_tcn = Sequential([
    tfa.layers.TCN(64, activation='relu', input_shape=(seq_length, 1)),
    Dense(1)
])

model_tcn.compile(optimizer='adam', loss='mse')

# Training
history_tcn = model_tcn.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_val, y_val)
)

# Prediction
y_pred_scaled_tcn = model_tcn.predict(X_val)
y_pred_tcn = scaler.inverse_transform(y_pred_scaled_tcn)
y_val_true = scaler.inverse_transform(y_val)

# Plot
val_index = df.index[train_size + seq_length:]
plt.figure(figsize=(15,5))
plt.plot(val_index, y_val_true, label='Validation')
plt.plot(val_index, y_pred_tcn, label='Forecast (TCN)')
plt.xlabel('Date')
plt.ylabel('Temperature (°C)')
plt.title('Hourly TCN Forecast vs Actual (t2m)')
plt.legend()
plt.show()

# Evaluation
mae_tcn = mean_absolute_error(y_val_true, y_pred_tcn)
rmse_tcn = np.sqrt(mean_squared_error(y_val_true, y_pred_tcn))
print(f"MAE: {mae_tcn}")
print(f"RMSE: {rmse_tcn}")


c:\Users\MSI\anaconda3\envs\my-project\Lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
c:\Users\MSI\anaconda3\envs\my-project\Lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.19.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.

ModuleNotFoundError: No module named 'keras.src.engine'